# 최종과제: 고객 문의 응답 형식 LoRA 파인튜닝

쇼핑몰 고객 문의에 **정해진 형식으로 답하는 모델**을 Qwen2.5-0.5B-Instruct과 LoRA로 만듭니다.

## 과제 목표

아래 항목을 확인합니다.

- SFT 데이터를 모델이 학습할 수 있는 대화 형식으로 변환할 수 있는가  
- LoRA 설정과 `SFTTrainer`로 모델을 학습할 수 있는가  
- 프롬프트 방식과 파인튜닝 결과를 **정량 지표**로 비교할 수 있는가  
- LoRA 설정을 바꿨을 때의 차이와 모델의 한계를 설명할 수 있는가

## 과제 시나리오

쇼핑몰 고객센터에는 하루 수만 건의 문의가 들어옵니다. 상담원이 문의를 빨리 처리할 수 있도록, 모든 문의를 아래 **세 줄 형식**으로 1차 정리하는 모델이 필요합니다.

```text
분류: 배송
긴급도: 높음
답변: 파손된 상품을 받으셔서 정말 죄송합니다. 상자와 제품 사진을 보내주시면 ...
```

| 항목 | 허용 값 |
|------|---------|
| 분류 | `배송`, `교환`, `환불`, `결제`, `계정` |
| 긴급도 | `높음`, `보통`, `낮음` |
| 답변 | 고객에게 보낼 1~2문장 안내 |

고객 개인정보가 담긴 문의라 외부 API로 보낼 수 없고, 처리량이 많아 긴 프롬프트를 매번 붙이기도 부담스럽습니다.  
그래서 **작은 공개 모델에 LoRA로 형식을 학습**시키기로 했습니다. (1일차 6.2절 실무 판단 순서의 3번 상황입니다.)

학습 데이터는 `data/support_inquiries.csv`에 있는 **고객 문의 100건**입니다.

## 전체 흐름

<img src="image/final_assignment_flow.svg" width="880">

## 제출 형태

제출물은 **캡처 보고서 1개**입니다.

- 파일명: `수행평가 제출양식_본인이름.docx`  
    - 본인 이름으로 변경해서 제출합니다.  
    - 예시: `수행평가 제출양식_김민수.docx`  
- 보고서 분량: 1~3쪽  
- 실행한 노트북 파일은 선택 제출입니다.

보고서에는 아래 실행 결과를 넣습니다.

| 번호 | 필수 캡처 | 위치 |
|---|---|---|
| 1 | 학습 전 평가 결과 (문의만 / 형식 지시 프롬프트) | `3. 학습 전 평가` |
| 2 | 학습 파라미터 비율과 손실 기록 | `4. LoRA 설정과 학습` |
| 3 | 학습 전후 비교표 | `5. 학습 후 평가` |
| 4 | 설정을 바꿔 다시 학습한 결과 | `6. 설정 바꿔 다시 학습하기` |

`7. 캡처용 제출 전 확인` 셀을 실행하면 네 가지 결과를 한 번에 모아 볼 수 있습니다.

## 채점 기준

총점은 100점입니다. 수료 기준은 60점입니다.

| 항목 | 점수 | 채점 기준 |
|---|---:|---|
| 제출 형식 | 20점 | 지정된 파일명 형식의 DOCX 보고서 제출 |
| 학습 전 평가 | 20점 | 문의만 넣은 경우와 형식 지시 프롬프트를 넣은 경우의 결과 포함 |
| LoRA 학습 | 20점 | 학습 가능한 파라미터 비율과 epoch별 손실 기록 포함 |
| 학습 후 평가 | 20점 | 학습 전후 비교표 포함, 학습 후 형식 준수율 80% 이상 |
| 설정 비교 및 설명 | 20점 | 바꾼 설정과 지표 변화, 네 가지 설명 항목 작성 |

## 제출 전 확인

보고서에 아래 내용이 들어 있는지 확인하세요.

- 필수 캡처 4개  
- 선택한 LoRA 설정(`r`, `lora_alpha`, `target_modules`, 학습률, epoch)과 그 이유  
- 학습 전후에 무엇이 달라졌는지  
- `r`을 바꿨을 때 지표가 어떻게 달라졌는지  
- 학습된 모델의 한계와 보완 방법

## 1. 환경 준비

이 노트북은 2일차와 같이 **VRAM 8GB 이상의 NVIDIA GPU 1개**를 전제로 합니다. GPU가 없으면 Google Colab(T4 GPU)에서 열어 실행하세요.
학습을 두 번 하므로 전체 실행에 5분 정도 걸립니다.

In [ ]:
import re  # 모델 응답이 세 줄 형식을 지켰는지 정규식으로 검사합니다.
import warnings  # 경고 메시지를 제어합니다.
warnings.filterwarnings("ignore")  # 실습과 무관한 경고를 숨깁니다.

import pandas as pd  # CSV를 읽고 결과를 표로 정리합니다.
import torch  # 모델 계산에 사용합니다.
from datasets import Dataset  # 변환한 대화 데이터를 Trainer가 읽는 형식으로 만듭니다.
from transformers import AutoModelForCausalLM, AutoTokenizer  # 생성 모델과 토크나이저입니다.
from peft import LoraConfig  # LoRA를 어디에 얼마나 붙일지 정하는 설정입니다.
from trl import SFTConfig, SFTTrainer  # SFT 학습 설정과 학습을 실행하는 Trainer입니다.

device = "cuda"  # 2일차와 같이 GPU 1개를 사용합니다.
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"  # 1일차 5.5절, 2일차에서 쓴 것과 같은 모델입니다.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)  # 모델과 한 세트로 쓰는 토크나이저입니다.
MODEL_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16  # 2일차 1.1절과 같은 기준입니다.


def load_model():  # 6장에서 설정을 바꿔 다시 학습할 때 학습되지 않은 모델이 한 번 더 필요해 함수로 만들어 둡니다.
    return AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,  # 내려받을 모델 이름입니다.
        dtype=MODEL_DTYPE,  # 가중치를 읽을 정밀도입니다.
    ).to(device)  # 모델을 GPU로 옮깁니다.


model = load_model()  # 학습 전 평가에 사용할 기반 모델입니다.
print("준비 완료:", MODEL_NAME)

In [ ]:
# 문의 한 건을 넣어 모델 응답을 받는 함수입니다. 2일차 chat 함수에서 형식을 깨뜨릴 수 있는 repetition_penalty만 뺐습니다.
def chat(model, question, max_new_tokens=100):
    inputs = tokenizer.apply_chat_template(  # 문의를 Qwen이 학습한 채팅 형식으로 바꿉니다.
        [{"role": "user", "content": question}],  # 사용자 메시지 한 개입니다.
        add_generation_prompt=True,  # 모델 답변이 시작될 위치까지 붙입니다.
        return_tensors="pt",  # PyTorch 텐서로 받습니다.
        return_dict=True,  # input_ids와 attention_mask를 함께 받습니다.
    ).to(model.device)  # 입력을 모델과 같은 장치로 옮깁니다.
    model.eval()  # 추론 모드로 바꿉니다.
    with torch.inference_mode():  # 그래디언트를 계산하지 않아 메모리와 시간을 아낍니다.
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)  # 항상 같은 결과가 나오는 Greedy 생성입니다.
    return tokenizer.decode(output[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()  # 새로 생성된 부분만 문자열로 돌려줍니다.

## 2. 데이터 준비

### 2.1 데이터 확인

In [ ]:
df = pd.read_csv("data/support_inquiries.csv")  # 고객 문의 100건을 읽습니다.
print("전체 문의 수:", len(df))
display(pd.crosstab(df["category"], df["urgency"], margins=True, margins_name="합계"))  # 분류별·긴급도별 건수를 봅니다.
df.head(3)  # 열이 어떻게 생겼는지 확인합니다.

### 2.2 학습 형식으로 변환

`format_answer()`는 한 행을 **세 줄 형식 문자열**로 만들어 줍니다. 이 함수는 그대로 쓰고, `to_conversation()`의 `None` 두 개만 채우세요.

2일차 2.5절에서 본 **Prompt-Completion 대화 형식**과 같습니다.

| 자리 | 넣을 값 |
|------|---------|
| `prompt` | 고객이 보낸 문의 문장 → `row["inquiry"]` |
| `completion` | 모델이 내놓아야 하는 세 줄 형식 문자열 → `format_answer(row)` |

In [ ]:
# 한 행을 세 줄 형식 문자열로 만듭니다. 이 함수는 그대로 사용합니다.
def format_answer(row):
    return f"분류: {row['category']}\n긴급도: {row['urgency']}\n답변: {row['answer']}"


# None 두 개를 채워 대화 형식 변환 함수를 완성하세요.
def to_conversation(row):
    return {
        "prompt": [{"role": "user", "content": None}],  # 고객이 보낸 문의 문장
        "completion": [{"role": "assistant", "content": None}],  # 모델이 내놓아야 하는 세 줄 형식 문자열
    }


print(to_conversation(df.iloc[0]))  # 첫 번째 문의가 제대로 바뀌었는지 확인합니다.

### 2.3 평가 데이터 분리

**분류마다 2건씩, 모두 10건**을 평가 데이터로 빼고 나머지 90건을 학습에 씁니다.
이 과제는 노트북을 처음부터 끝까지 한 번 실행하고 끝내므로, 검증 데이터를 따로 두지 않고 평가 10건을 학습 중 손실 확인에도 함께 씁니다.

평가 10건은 분류가 2건씩이고 긴급도는 보통 4건·높음 3건·낮음 3건입니다.
그래서 **항상 같은 값만 답하는 모델도 분류 정확도 20%, 긴급도 정확도 40%까지는 나옵니다.** 이 두 숫자가 학습 효과를 판단하는 기준선입니다.

이 셀은 수정하지 않습니다.

In [ ]:
test_df = df.groupby("category", group_keys=False).sample(n=2, random_state=1)  # 분류마다 2건씩 뽑습니다. 긴급도가 골고루 섞이는 조합이라 random_state를 1로 뒀습니다.
train_df = df.drop(test_df.index)  # 평가로 뽑힌 행을 뺀 나머지 90건이 학습 데이터입니다.

train_ds = Dataset.from_list([to_conversation(row) for _, row in train_df.iterrows()])  # 학습에 쓸 대화 데이터입니다.
test_ds = Dataset.from_list([to_conversation(row) for _, row in test_df.iterrows()])  # 학습 중 손실을 확인할 때 씁니다.
print(f"학습 {len(train_ds)}건 / 평가 {len(test_ds)}건")
print("\n[학습 예시의 답변]")
print(train_ds[0]["completion"][0]["content"])  # 모델이 따라 쓰도록 학습할 정답입니다.

## 3. 학습 전 평가

평가 10건으로 **학습하지 않은 모델**을 두 가지 방식으로 먼저 측정합니다. 이 절의 셀은 수정하지 않습니다.

| 방식 | 모델에 넣는 입력 |
|------|------------------|
| 문의만 | 고객 문의 한 문장 (학습한 모델이 받게 될 입력과 같습니다) |
| 형식 지시 프롬프트 | 세 줄 형식을 설명하는 지시문 + 고객 문의 |

두 번째 방식은 1일차 6.2절의 첫 질문인 **"프롬프트로 해결되는가?"** 를 실제로 확인하는 과정입니다. 여기서 충분히 잘 된다면 파인튜닝은 필요하지 않습니다.

| 지표 | 계산 방법 |
|------|-----------|
| 형식 준수율 | 응답이 `분류: {허용 값}` / `긴급도: {허용 값}` / `답변: ...` 세 줄 형식을 지킨 비율 |
| 분류 정확도 | 형식을 지키고 분류가 정답과 같은 비율 |
| 긴급도 정확도 | 형식을 지키고 긴급도가 정답과 같은 비율 |

세 지표는 모두 **형식을 지킨 응답에서만** 셉니다. 형식이 깨지면 분류와 긴급도를 꺼낼 수 없기 때문입니다.

In [ ]:
# 응답이 세 줄 형식을 지켰는지 검사하는 정규식입니다. 분류와 긴급도는 허용 값인지까지 함께 봅니다.
FORMAT_PATTERN = re.compile(r"^분류: (배송|교환|환불|결제|계정)\n긴급도: (높음|보통|낮음)\n답변: .+", re.S)

# 형식을 말로 지시하는 프롬프트입니다. 문의 문장 앞에 붙여서 넣습니다.
FORMAT_PROMPT = """고객 문의를 아래 세 줄 형식으로만 정리하라.
분류: 배송, 교환, 환불, 결제, 계정 중 하나
긴급도: 높음, 보통, 낮음 중 하나
답변: 고객에게 보낼 1~2문장 안내

문의: """


def evaluate_model(model, eval_df, prefix=""):  # prefix에 FORMAT_PROMPT를 주면 지시문을 붙여 평가합니다.
    outputs, fmt_ok, cat_ok, urg_ok = [], 0, 0, 0  # 응답 목록과 지표별 정답 개수입니다.
    for _, row in eval_df.iterrows():  # 평가 데이터를 한 건씩 모델에 넣습니다.
        answer = chat(model, prefix + row["inquiry"])  # 모델 응답을 받습니다.
        outputs.append(answer)
        match = FORMAT_PATTERN.match(answer)  # 세 줄 형식을 지켰는지 확인합니다.
        if match:  # 형식을 지킨 응답에서만 분류와 긴급도를 채점합니다.
            fmt_ok += 1
            cat_ok += match.group(1) == row["category"]  # 첫 줄의 분류가 정답과 같은지
            urg_ok += match.group(2) == row["urgency"]  # 둘째 줄의 긴급도가 정답과 같은지
    n = len(eval_df)
    return {"형식 준수율": fmt_ok / n, "분류 정확도": cat_ok / n, "긴급도 정확도": urg_ok / n}, outputs


before_metrics, before_outputs = evaluate_model(model, test_df)  # 문의만 넣었을 때
prompt_metrics, prompt_outputs = evaluate_model(model, test_df, prefix=FORMAT_PROMPT)  # 형식을 지시했을 때

print("[학습 전 평가 결과]")
display(pd.DataFrame({"문의만": before_metrics, "형식 지시 프롬프트": prompt_metrics}).map(lambda v: f"{v:.0%}"))

sample = test_df.iloc[0]["inquiry"]  # 입력 토큰 수를 비교해 볼 문의 한 건입니다.
only_tokens = tokenizer.apply_chat_template([{"role": "user", "content": sample}], add_generation_prompt=True)  # 문의만 넣었을 때 모델이 받는 토큰입니다.
prompt_tokens = tokenizer.apply_chat_template([{"role": "user", "content": FORMAT_PROMPT + sample}], add_generation_prompt=True)  # 지시문까지 붙였을 때입니다.
print(f"입력 토큰: 문의만 {len(only_tokens)}개 -> 지시문 포함 {len(prompt_tokens)}개")
print(f"\n[형식 지시 프롬프트를 넣은 응답]\n{prompt_outputs[0][:200]}")

## 4. LoRA 설정과 학습

아래 두 셀의 `None`을 채우세요. 2일차 4.6절의 하이퍼파라미터 가이드를 참고합니다.

| 설정 | 권장 값 |
|------|---------|
| `r` | 8 또는 16 |
| `lora_alpha` | `r`의 2배 |
| `target_modules` | Attention 4개 + FFN 3개, 모두 7개 (이름은 코드 주석에 있습니다) |
| `learning_rate` | 2e-4 ~ 3e-4 |
| `num_train_epochs` | 3 ~ 5 (학습 데이터가 90건으로 적어 여러 번 반복합니다) |

학습 후 형식 준수율이 80%에 못 미치면 epoch이나 학습률을 조금 올려 다시 실행해 보세요.

In [ ]:
# None 세 개를 채워 LoRA 설정을 완성하세요.
lora_config = LoraConfig(
    r=None,  # LoRA 랭크입니다. 8 또는 16
    lora_alpha=None,  # 배율입니다. r의 2배
    lora_dropout=0.05,  # LoRA 경로의 dropout입니다. 그대로 둡니다.
    target_modules=None,  # LoRA를 붙일 선형층 이름 리스트입니다. ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
    task_type="CAUSAL_LM",  # 다음 토큰을 예측하는 모델임을 알려 줍니다.
)

In [ ]:
# None 두 개를 채워 학습 설정을 완성하세요. 나머지는 2일차 5.3절과 같습니다.
sft_config = SFTConfig(
    output_dir="outputs/support-lora",  # 학습 결과를 저장할 폴더입니다.
    num_train_epochs=None,  # 학습 데이터를 몇 번 반복할지 정합니다. 3~5
    learning_rate=None,  # LoRA 가중치의 학습률입니다. 2e-4 ~ 3e-4
    per_device_train_batch_size=1,  # 한 번에 학습하는 예시 수입니다.
    gradient_accumulation_steps=8,  # 8스텝을 모아 한 번 업데이트합니다.
    lr_scheduler_type="cosine",  # 학습이 진행될수록 학습률을 줄입니다.
    warmup_ratio=0.05,  # 처음에는 학습률을 서서히 올립니다.
    max_length=256,  # 템플릿을 적용한 뒤의 최대 토큰 수입니다.
    completion_only_loss=True,  # 질문 구간은 빼고 답변 구간만 학습합니다.
    eval_strategy="epoch",  # epoch마다 평가 손실을 기록합니다.
    logging_strategy="epoch",  # epoch마다 학습 손실을 기록합니다.
    save_strategy="no",  # 중간 체크포인트는 저장하지 않습니다.
    bf16=MODEL_DTYPE == torch.bfloat16,  # 모델과 같은 정밀도로 계산합니다.
    fp16=MODEL_DTYPE == torch.float16,  # BF16을 지원하지 않는 GPU는 FP16을 씁니다.
)

In [ ]:
trainer = SFTTrainer(  # 모델·데이터·설정을 하나의 학습 객체로 묶습니다.
    model=model,  # LoRA를 붙일 기반 모델입니다.
    args=sft_config,  # 위에서 만든 학습 설정입니다.
    train_dataset=train_ds,  # 학습 데이터 90건입니다.
    eval_dataset=test_ds,  # epoch마다 손실을 확인할 데이터 10건입니다.
    processing_class=tokenizer,  # 템플릿 적용과 토큰화를 맡습니다.
    peft_config=lora_config,  # SFTTrainer가 이 설정대로 LoRA를 붙입니다.
)
trainer.model.print_trainable_parameters()  # 전체 중 몇 퍼센트를 학습하는지 출력합니다.
trainer.train()  # 학습을 실행합니다. 1분 30초 정도 걸립니다.

loss_log = pd.DataFrame(trainer.state.log_history).groupby("epoch")[["loss", "eval_loss"]].first().round(3)  # epoch별 학습 손실과 평가 손실을 표로 모읍니다.
loss_log

## 5. 학습 후 평가

학습 전과 **같은 평가 데이터, 같은 지표**로 다시 평가해 세 방식을 나란히 비교합니다. 이 셀은 수정하지 않습니다.

In [ ]:
lora_model = trainer.model  # 학습이 끝난 LoRA 모델입니다.
after_metrics, after_outputs = evaluate_model(lora_model, test_df)  # 학습 전과 같은 데이터, 같은 지표로 측정합니다.

comparison = pd.DataFrame({  # 세 방식을 한 표에 모읍니다.
    "학습 전 (문의만)": before_metrics,
    "학습 전 (프롬프트)": prompt_metrics,
    f"학습 후 (r={lora_config.r})": after_metrics,
}).map(lambda v: f"{v:.0%}")
display(comparison)

samples = pd.DataFrame({  # 어떤 문의에서 틀렸는지 직접 확인합니다.
    "문의": test_df["inquiry"].values,
    "정답": (test_df["category"] + " / " + test_df["urgency"]).values,
    "학습 후 응답": [answer.replace("\n", " / ") for answer in after_outputs],  # 세 줄을 한 줄로 이어 표에서 읽기 쉽게 만듭니다.
})
pd.set_option("display.max_colwidth", 150)  # 응답이 잘리지 않게 표시 폭을 늘립니다.
samples

지표별로 무엇이 학습됐는지 갈립니다.

**형식**: 지시문만으로는 절반 정도만 지켜집니다. 지시문을 그대로 베껴 `분류: 배송, 교환, 환불, 결제, 계정 중 하나`라고 답하는 실패가 많습니다.
학습 후에는 대부분 형식을 지키지만, `분류: 마케팅`처럼 **허용 값에 없는 분류를 만들어 내는** 응답이 남기도 합니다.

**분류**: 기준선 20%를 크게 넘습니다. 세 지표 중 가장 잘 학습되는 항목입니다.

**긴급도**: 기준선 40%를 조금 넘는 수준에 머무르기 쉽습니다. 위 표에서 모델이 쓴 긴급도를 세어 보세요. 대부분 `보통`이고 `높음`은 한 번도 쓰지 않을 수 있습니다.
학습 데이터에서 `높음`이 18건뿐이라 판단 근거가 부족한 것이 한 가지 원인입니다.
긴급한 문의를 낮게 판단하는 오류는 서비스에서 치명적이므로, 해당 유형의 데이터를 보강하거나 키워드 규칙으로 한 번 더 점검하는 방법을 함께 씁니다.

**답변 내용**: 0.5B 모델의 한계로 없는 정책이나 날짜를 지어냅니다. 이 과제로 학습된 것은 **출력 형식과 분류 기준**이고, 답변의 사실성은 학습 대상이 아니었습니다.
실제 서비스라면 답변은 RAG로 정책 문서를 참고하게 하거나 사람이 검토하는 단계를 두어야 합니다.

위 손실 표에서 **평가 손실이 더 이상 줄지 않거나 다시 오르는 지점**이 있는지도 확인하세요. 거기서부터는 학습 데이터를 외우기 시작한 것(과적합)입니다.

## 6. 설정 바꿔 다시 학습하기

LoRA 설정이 결과를 얼마나 바꾸는지 직접 확인합니다. **`r`만 더 작은 값으로 바꿔** 같은 데이터로 한 번 더 학습하고, 앞의 비교표에 열을 하나 더 붙입니다.

`r`을 줄이면 학습하는 파라미터가 줄어듭니다. 세 지표가 함께 떨어지는지, 아니면 어떤 항목만 떨어지는지 확인하는 것이 이 절의 목적입니다.

아래 `None` 두 개를 채우세요. 학습에 1분 30초 정도 걸립니다.

In [ ]:
# None 두 개를 채워 두 번째 LoRA 설정을 만드세요. r만 줄이고 나머지는 4장과 같게 둡니다.
small_config = LoraConfig(
    r=None,  # 4장에서 쓴 r보다 작은 값입니다. 예: 4
    lora_alpha=None,  # 위 r의 2배
    lora_dropout=0.05,
    target_modules=lora_config.target_modules,  # 4장과 같은 층에 붙입니다.
    task_type="CAUSAL_LM",
)

small_trainer = SFTTrainer(
    model=load_model(),  # 4장에서 학습한 모델을 다시 쓰면 안 되므로 새 모델을 불러옵니다.
    args=sft_config,  # 학습 설정은 4장과 똑같이 두고 LoRA 설정만 바꿉니다.
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    peft_config=small_config,
)
small_trainer.model.print_trainable_parameters()  # 학습 파라미터가 얼마나 줄었는지 봅니다.
small_trainer.train()

small_metrics, _ = evaluate_model(small_trainer.model, test_df)  # 같은 평가 데이터, 같은 지표로 측정합니다.
final_comparison = comparison.copy()  # 5장 비교표에 열을 하나 더 붙입니다.
final_comparison[f"학습 후 (r={small_config.r})"] = pd.Series(small_metrics).map(lambda v: f"{v:.0%}")
final_comparison

`r`을 줄이면 학습 파라미터는 눈에 띄게 줄지만, 세 지표가 모두 같은 비율로 떨어지지는 않습니다.
형식처럼 **반복되는 고정 규칙**은 작은 `r`로도 충분히 학습되는 반면, 분류처럼 **문장의 의미를 구분해야 하는 항목**이 먼저 떨어지는 경향이 있습니다.

실무에서는 이 점을 이용해 "필요한 만큼만 큰 `r`"을 찾습니다. `r`이 작을수록 어댑터 파일과 메모리가 줄기 때문입니다.
보고서에는 두 설정의 **학습 파라미터 수와 지표를 함께** 적고, 어느 쪽을 쓰겠는지 이유와 함께 쓰세요.

## 7. 캡처용 제출 전 확인

아래 셀은 제출 보고서에 넣을 실행 결과를 한곳에 모아 보여줍니다.  
실행 후 출력 영역을 캡처해서 보고서에 붙여도 됩니다.

In [ ]:
print("[캡처 1] 학습 전 평가 결과")
print(pd.DataFrame({"문의만": before_metrics, "형식 지시 프롬프트": prompt_metrics}).map(lambda v: f"{v:.0%}").to_string())

print("\n[캡처 2] 학습 파라미터 비율과 손실 기록")
trainable, total = trainer.model.get_nb_trainable_parameters()  # LoRA로 학습한 파라미터 수와 전체 파라미터 수입니다.
print(f"  r={lora_config.r}, lora_alpha={lora_config.lora_alpha}, 학습률={sft_config.learning_rate}, epoch={sft_config.num_train_epochs}")
print(f"  학습 파라미터: {trainable:,} / 전체 {total:,} ({trainable / total:.2%})")
print(loss_log.to_string())

print("\n[캡처 3] 학습 전후 비교표")
print(comparison.to_string())

print("\n[캡처 4] 설정을 바꾼 결과")
small_trainable, _ = small_trainer.model.get_nb_trainable_parameters()  # 두 번째 설정의 학습 파라미터 수입니다.
print(f"  학습 파라미터: r={lora_config.r} -> {trainable:,}개 / r={small_config.r} -> {small_trainable:,}개")
print(final_comparison.to_string())

## 8. 보고서에 넣을 설명 작성

제출 보고서 마지막에 아래 항목을 3~5줄로 작성하세요. 채점자는 이 부분을 보고 설명 점수를 부여합니다.

- 선택한 LoRA 설정과 그 이유:  
- 학습 전후에 달라진 점 (지표와 응답 예시 기준):  
- 이 문제에 프롬프트 대신 파인튜닝을 사용한 이유:  
- `r`을 바꿨을 때의 변화와 학습된 모델의 한계·보완 방법:

작성 예시:

```text
r=16, lora_alpha=32로 Attention과 FFN의 선형층 7개에 LoRA를 적용했고, 전체 파라미터의 약 1.75%만 학습했다.
형식 지시 프롬프트만 썼을 때는 형식 준수율 50%, 분류 40%, 긴급도 10%였지만 학습 후에는 각각 90%, 90%, 50%가 되었다.
지시문은 입력 토큰을 49개에서 125개로 늘리면서도 선택지를 그대로 베끼는 응답이 많았다. 처리량이 많은 업무이므로 작은 모델에 형식을 학습시키는 편이 비용과 일관성 면에서 유리하다.
r을 4로 줄이자 학습 파라미터는 1/4로 줄고 형식 준수율은 그대로였지만 분류 정확도가 90%에서 70%로 떨어졌다. 답변 내용은 모델이 작아 사실과 다를 수 있으므로 RAG와 상담원 검토를 함께 적용해야 한다.
```